Датасет был взят по ссылке https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques.

В датасете есть характеристики домов, которые продаются. В характеристики входят улица, на которой расположен дом, его размеры, присутствие/отсутсвие аллеи, какой формы дом, какие услуги ЖКХ в нем есть и многое многое другое.

Цель этого датасета - суметь предугадать стоимость домов.


Основной файл с полной информацией - train.csv.

Файл test.csv то же самое, что и train, но без столба цен.
Файл data_description - вспомогательный файл, в котором прописаны расшифровки сокращений.



A) Быстрый обзор данных (Pandas)

In [ ]:
import pandas as pd

df = pd.read_csv('train.csv')
print("--- Первые 5 строк ---")
print(df.head())

In [ ]:
print("--- Последние 5 строк ---")
print(df.tail())

In [ ]:
print(f"\nРазмер таблицы (строки, колонки): {df.shape}")
print("\n--- Техническая информация ---")
df.info()

In [ ]:
print("--- Статистика числовых признаков ---")
print(df.describe())
print("\n--- Статистика текстовых признаков ---")
print(df.describe(include=['object', 'string']))  #используется ['object', 'string'], т.к. при обычном 'object' возникает предупреждение о возможной несовместимости в будущих версиях pandas

In [ ]:
print("--- Пропущенные значения в колонках ---")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0]) # Выведет только колонки с дырками

print("\n--- Типы данных по столбцам ---")
print(df.dtypes.value_counts())

duplicates = df.duplicated().sum()
print(f"\nКоличество полных дубликатов строк: {duplicates}")


😎 Пропуски и очистка

In [ ]:
#Использовать dropna() в данном случае не слишком уместно, т.к. данных не сотни тысяч строк, а пропусков немалое количество. При чистке таким способо будет выкинуто слишком много данных.

#Поэтому буду использовать fillna(). Если ячейка напротив PoolQC пустая - это вполне нормально, ведь бассейна просто может и не быть. В таком случае, заполню ячейку NaN`ом.

cols_to_fix = ['PoolQC', 'MiscFeature', 'Alley', 'Fence'] #в них наибольшее кол-во пропусков
for col in cols_to_fix:
    df[col] = df[col].fillna("NaN")

electrical_mode = df['Electrical'].mode()[0]

#Если мы не знаем тип проводки, логично предположить, что там стоит самый популярный стандарт для таких домов.
df['Electrical'] = df['Electrical'].fillna(electrical_mode)

C) Расширенная статистика

In [ ]:
numeric_cols = df.select_dtypes(include=['number'])
stats = numeric_cols.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T

stats['mode'] = numeric_cols.mode().iloc[0]
stats['variance'] = numeric_cols.var() #Показывает, насколько сильно разбросаны цены
stats['skewness'] = numeric_cols.skew() #Показывает "гору" дешевых домов, которая плавно перейдет в спуск к дорогим домам
stats['kurtosis'] = numeric_cols.kurt() #Показывает выбросы в ценах(в данном случае - резкое подаражание на фоне дешевых домов)

stats = stats.rename(columns={'50%': 'median'})

print(stats[['min', 'max', 'mean', 'median', 'mode', 'variance', 'skewness', 'kurtosis']].head())

D) Фичи: Энкодинг и Инжиниринг (Feature Engineering)

In [ ]:
# Применяем для колонок с малым количеством уникальных значений
df_ohe = pd.get_dummies(df, columns=['MSZoning', 'Street'], prefix='OHE')

# Позволяет сжать много категорий в фиксированное число колонок
from sklearn.feature_extraction import FeatureHasher
hasher = FeatureHasher(n_features=5, input_type='string')
hashed_features = hasher.transform(df['Neighborhood'].astype(str).apply(lambda x: [x])).toarray()
hashed_df = pd.DataFrame(hashed_features, columns=[f'neigh_hash_{i}' for i in range(5)])
df_final = pd.concat([df_ohe, hashed_df], axis=1)
#print(df_final.head())

#Общая площадь
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

#Возраст дома на момент продажи
df['HouseAge'] = df['YrSold'] - df['YearBuilt']

E) Визуализация

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))
# Считаем корреляцию для топ-10 признаков
numeric_df = df.select_dtypes(include=['number'])
top_corr = numeric_df.corr()['SalePrice'].sort_values(ascending=False).head(10).index
sns.heatmap(df[top_corr].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Топ-10 корреляций с ценой продажи')
plt.show()

По хорошему, SalePrice нужно предугадвать, поэтому она не в учет.
Наибольшие изменения вносят OverallQual и TotalSF (ну, как бы почти самое важно для домов)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Гистограмма цен
sns.histplot(df['SalePrice'], kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Распределение цен (skewness)')

# Boxplot: Качество дома / Цена
sns.boxplot(x='OverallQual', y='SalePrice', data=df, ax=axes[1])
axes[1].set_title('Поиск выбросов: Качество / Цена')

plt.show()

In [ ]:
import plotly.express as px

fig = px.scatter(df, x="GrLivArea", y="SalePrice",
                 color="OverallQual",
                 hover_data=['Neighborhood', 'YearBuilt'],
                 title="Зависимость цены от площади и района",
                 template="plotly_dark")

fig.update_layout(plot_bgcolor='black', paper_bgcolor='black')

fig.show()

In [ ]:
plt.figure(figsize=(15, 6))
sns.countplot(data=df, x='Neighborhood', order=df['Neighborhood'].value_counts().index)
plt.xticks(rotation=45)
plt.title('Количество проданных домов по районам')
plt.show()

F) Итоговые выводы (Markdown)



* Что я понял:
1) В среденем домики стоят 130-200к
2) OverallQual и TotalSF - главные факторы для цены
3) Новые дома стоят явно больше, нежели старичик
4) У большого кол-ва домов нет бассейнов D:
5) Есть "элитные" райончики, где цены взлетают под 2-3 раза (например, NoRidge)
6) Тип проводки не указали только в одном поле :/
7) Ни один дом не совпал с другим полностью (для домов как бы это нормально, но все же)

* Гипотезы/наблюдения
1) Природа выбросов: На графике GrLivArea / SalePrice есть два дома с огромной площадью ( >4000 кв.), но низкой ценой. Скорее всего, дома еле стоят(в смысле чуть не разваливаются). Их стоит удалить перед обучением.
2) Качество кухни (KitchenQual) может иногда перевешивать другие хар-тики дома.
3) Возможно, цены зависят от месяца продажи (MoSold). Дома, проданные весной и в начале лета, стоят дороже.

* Что бы вы сделали дальше
Прошерстил бы список моделей и выбрал бы те, что хорошо адаптируются к примененным мною OHE и Hasher`у (например, Ridge).
Ну и если брать test.cvs как основной, то обучил бы модель предсказывать цены на дома.

* Какие подсказки брал у AI и что в итоге проверял/дописывал руками
Просил показать а как пишуться команды, которые делают, что просит условие, а потом переписывал их под те данные, которые
мог найти в таблицах.
Принцип такой: показали - повторил на своем.